In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [8]:
import sys

PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"
sys.path.append(PROJECT_DIR)

In [11]:
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"

!jupyter nbconvert --to python "$PROJECT_DIR/preprocessing.ipynb"
!jupyter nbconvert --to python "$PROJECT_DIR/models.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.ipynb to python
[NbConvertApp] Writing 5391 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.py
[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/models.ipynb to python
[NbConvertApp] Writing 3362 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/models.py


In [17]:
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mlflow
import os
from typing import Dict, Tuple

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

from preprocessing import prepare_dataset
from models import build_model

In [18]:
# --------------------------------------------------------------------- #
# Configuracion de rutas para Google Colab
# --------------------------------------------------------------------- #
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"
DATA_PATH = f"{PROJECT_DIR}/Data/02-15-2018.csv"
RESULTS_DIR = f"{PROJECT_DIR}/results"
MLFLOW_DB_PATH = f"{PROJECT_DIR}/mlflow.db"

os.makedirs(RESULTS_DIR, exist_ok=True)
# --------------------------------------------------------------------- #
# Utilidades
# --------------------------------------------------------------------- #
def validate_paths() -> None:
    """Valida que las rutas necesarias existan antes de entrenar."""
    required_files = [
        DATA_PATH,
        f"{PROJECT_DIR}/preprocessing.py",
        f"{PROJECT_DIR}/models.py",
    ]

    missing = [path for path in required_files if not os.path.exists(path)]

    if missing:
        message = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "No se encontraron los siguientes archivos necesarios:\n"
            f"{message}\n\n"
            "Verifica que Google Drive este montado y que la estructura sea:\n"
            f"{PROJECT_DIR}/Data/02-15-2018.csv\n"
            f"{PROJECT_DIR}/preprocessing.py\n"
            f"{PROJECT_DIR}/models.py"
        )

In [19]:
# --------------------------------------------------------------------- #
# Configuracion experimental
# --------------------------------------------------------------------- #
SEEDS = [42, 43, 44, 45]
MODEL_NAMES = ["MLP", "LSTM", "Autoencoder"]
EPOCHS = {"MLP": 40, "LSTM": 10, "Autoencoder": 60}
LR = {"MLP": 1e-3, "LSTM": 1e-3, "Autoencoder": 1e-3}
BATCH_SIZE = 256
SPLIT_SEED = 42
MLFLOW_EXPERIMENT = "SOC_Autonomous_Threat_Hunting_DL"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --------------------------------------------------------------------- #
# Utilidades
# --------------------------------------------------------------------- #
def validate_paths() -> None:
    """Valida que las rutas necesarias existan antes de entrenar."""
    required_files = [
        DATA_PATH,
        f"{PROJECT_DIR}/preprocessing.py",
        f"{PROJECT_DIR}/models.py",
    ]

    missing = [path for path in required_files if not os.path.exists(path)]

    if missing:
        message = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "No se encontraron los siguientes archivos necesarios:\n"
            f"{message}\n\n"
            "Verifica que Google Drive este montado y que la estructura sea:\n"
            f"{PROJECT_DIR}/Data/02-15-2018.csv\n"
            f"{PROJECT_DIR}/preprocessing.py\n"
            f"{PROJECT_DIR}/models.py"
        )


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=DEVICE)


def safe_roc_auc(y_true, y_score) -> float:
    """Evita error cuando y_true tiene una sola clase."""
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return roc_auc_score(y_true, y_score)


def safe_pr_auc(y_true, y_score) -> float:
    """Evita error cuando y_true tiene una sola clase."""
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return average_precision_score(y_true, y_score)


def compute_metrics(y_true, y_score, y_pred) -> Dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_roc_auc(y_true, y_score),
        "pr_auc": safe_pr_auc(y_true, y_score),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def best_threshold_by_f1(y_true, y_score) -> float:
    """Busca el umbral que maximiza F1 en el set de validacion."""
    thresholds = np.unique(y_score)

    if len(thresholds) > 200:
        thresholds = np.quantile(y_score, np.linspace(0, 1, 200))

    best_t = 0.5
    best_f1 = -1.0

    for threshold in thresholds:
        pred = (y_score >= threshold).astype(int)
        score = f1_score(y_true, pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_t = threshold

    return float(best_t)


# --------------------------------------------------------------------- #
# Entrenamiento supervisado: MLP y LSTM
# --------------------------------------------------------------------- #
def train_supervised(model_name: str, data: Dict, seed: int) -> Tuple[nn.Module, Dict]:
    set_seed(seed)

    model = build_model(model_name, data["n_features"]).to(DEVICE)

    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val = data["X_val"], data["y_val"]
    X_test, y_test = data["X_test"], data["y_test"]

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR[model_name], weight_decay=1e-5)

    X_train_t = to_tensor(X_train)
    y_train_t = to_tensor(y_train)
    n = X_train_t.shape[0]

    epochs = EPOCHS[model_name]
    rng = np.random.default_rng(seed)

    for epoch in range(epochs):
        model.train()
        perm = rng.permutation(n)
        epoch_loss = 0.0

        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb = X_train_t[idx]
            yb = y_train_t[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(idx)

        mlflow.log_metric("train_loss", epoch_loss / n, step=epoch)

    def predict_scores(X):
        model.eval()
        with torch.no_grad():
            logits = model(to_tensor(X))
            return torch.sigmoid(logits).cpu().numpy()

    val_scores = predict_scores(X_val)
    threshold = best_threshold_by_f1(y_val, val_scores)

    test_scores = predict_scores(X_test)
    test_pred = (test_scores >= threshold).astype(int)

    metrics = compute_metrics(y_test, test_scores, test_pred)
    metrics["threshold"] = float(threshold)

    return model, metrics


# --------------------------------------------------------------------- #
# Entrenamiento no supervisado: Autoencoder
# --------------------------------------------------------------------- #
def train_autoencoder(data: Dict, seed: int) -> Tuple[nn.Module, Dict]:
    """
    Entrena el Autoencoder con la clase dominante del set de entrenamiento.

    Recomendacion metodologica:
    - En un SOC real, lo ideal es entrenarlo con trafico benigno.
    - En este pipeline, se mantiene la logica original: usar la clase dominante.
    - Si la clase dominante es ataque, el Autoencoder aprende ese patron dominante
      y marca como anomalos los flujos que se alejan de el.
    """
    set_seed(seed)

    model = build_model("Autoencoder", data["n_features"]).to(DEVICE)

    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val = data["X_val"], data["y_val"]
    X_test, y_test = data["X_test"], data["y_test"]

    dominant_label = pd.Series(y_train).value_counts().idxmax()
    X_train_dom = X_train[y_train == dominant_label]

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR["Autoencoder"], weight_decay=1e-5)

    X_train_t = to_tensor(X_train_dom)
    n = X_train_t.shape[0]
    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS["Autoencoder"]):
        model.train()
        perm = rng.permutation(n)
        epoch_loss = 0.0

        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb = X_train_t[idx]

            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(idx)

        mlflow.log_metric("train_recon_loss", epoch_loss / n, step=epoch)

    def recon_error(X):
        model.eval()
        with torch.no_grad():
            X_t = to_tensor(X)
            recon = model(X_t)
            err = torch.mean((recon - X_t) ** 2, dim=1)
            return err.cpu().numpy()

    val_err = recon_error(X_val)
    val_score_dominant = -val_err
    y_val_dominant = (y_val == dominant_label).astype(int)
    threshold = best_threshold_by_f1(y_val_dominant, val_score_dominant)

    test_err = recon_error(X_test)
    test_score_dominant = -test_err
    test_pred_dominant = (test_score_dominant >= threshold).astype(int)

    # Reexpresar en terminos de la clase positiva real: y = 1 = Ataque
    if dominant_label == 1:
        y_score_pos = test_score_dominant
        y_pred_pos = test_pred_dominant
    else:
        y_score_pos = -test_score_dominant
        y_pred_pos = 1 - test_pred_dominant

    metrics = compute_metrics(y_test, y_score_pos, y_pred_pos)
    metrics["threshold"] = float(threshold)
    metrics["dominant_label_trained_on"] = int(dominant_label)

    return model, metrics


# --------------------------------------------------------------------- #
# Resumen estadistico
# --------------------------------------------------------------------- #
def build_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    metrics_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
    rows = []

    for model_name, group in results_df.groupby("model"):
        row = {"Modelo": model_name, "N_corridas": len(group)}

        for metric in metrics_cols:
            row[f"{metric}_mean"] = round(group[metric].mean(), 4)
            row[f"{metric}_std"] = round(group[metric].std(ddof=1), 4)

        rows.append(row)

    return pd.DataFrame(rows)


# --------------------------------------------------------------------- #
# Orquestacion principal
# --------------------------------------------------------------------- #
def run_all():
    validate_paths()

    print("Dispositivo:", DEVICE)
    print("Proyecto:", PROJECT_DIR)
    print("Dataset:", DATA_PATH)
    print("Resultados:", RESULTS_DIR)

    mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    data = prepare_dataset(DATA_PATH, random_state=SPLIT_SEED)

    print(
        f"N features: {data['n_features']} | "
        f"Train: {data['X_train'].shape} | "
        f"Val: {data['X_val'].shape} | "
        f"Test: {data['X_test'].shape}"
    )

    if "target_mapping" in data:
        print("Mapeo del target:", data["target_mapping"])

    if "original_label_counts" in data:
        print("Distribucion original de labels:")
        for label, count in data["original_label_counts"].items():
            print(f"  {label}: {count}")

    if "detected_attack_labels" in data:
        print("Labels considerados como ataque:")
        for label in data["detected_attack_labels"]:
            print(f"  - {label}")

    print(f"Distribucion train codificada (0=Benign, 1=Ataque): {data['class_counts_train']}")

    all_results = []

    for model_name in MODEL_NAMES:
        for seed in SEEDS:
            run_name = f"{model_name}_seed{seed}"

            with mlflow.start_run(run_name=run_name):
                mlflow.log_params({
                    "model": model_name,
                    "seed": seed,
                    "epochs": EPOCHS[model_name],
                    "lr": LR[model_name],
                    "batch_size": BATCH_SIZE,
                    "n_features": data["n_features"],
                    "split_seed": SPLIT_SEED,
                    "positive_class": "Attack / Non-Benign",
                    "data_path": DATA_PATH,
                    "project_dir": PROJECT_DIR,
                })

                t0 = time.time()

                if model_name == "Autoencoder":
                    model, metrics = train_autoencoder(data, seed)
                else:
                    model, metrics = train_supervised(model_name, data, seed)

                elapsed = time.time() - t0
                metrics["train_time_sec"] = elapsed

                mlflow.log_metrics({
                    key: value
                    for key, value in metrics.items()
                    if isinstance(value, (int, float)) and not pd.isna(value)
                })

                artifact_path = f"{RESULTS_DIR}/model_{run_name}.pt"
                torch.save(model.state_dict(), artifact_path)
                mlflow.log_artifact(artifact_path)

                print(
                    f"[{model_name} | seed={seed}] "
                    f"F1={metrics['f1']:.4f} "
                    f"PR-AUC={metrics['pr_auc']:.4f} "
                    f"ROC-AUC={metrics['roc_auc']:.4f} "
                    f"Recall={metrics['recall']:.4f} "
                    f"({elapsed:.1f}s)"
                )

                row = {"model": model_name, "seed": seed}
                row.update(metrics)
                all_results.append(row)

    results_df = pd.DataFrame(all_results)

    all_runs_path = f"{RESULTS_DIR}/all_runs.csv"
    summary_csv_path = f"{RESULTS_DIR}/summary_statistics.csv"
    summary_json_path = f"{RESULTS_DIR}/summary_statistics.json"

    results_df.to_csv(all_runs_path, index=False)

    summary = build_summary_table(results_df)
    summary.to_csv(summary_csv_path, index=False)

    with open(summary_json_path, "w", encoding="utf-8") as file:
        json.dump(summary.to_dict(orient="records"), file, indent=2, ensure_ascii=False)

    print("\n=== CUADRO ESTADISTICO (media +/- std sobre corridas) ===")
    print(summary.to_string(index=False))

    print("\nArchivos generados:")
    print("-", all_runs_path)
    print("-", summary_csv_path)
    print("-", summary_json_path)
    print(f"- modelos .pt en {RESULTS_DIR}")

    return results_df, summary


if __name__ == "__main__":
    run_all()

Dispositivo: cpu
Proyecto: /content/drive/MyDrive/Trabajo_Cualitativo
Dataset: /content/drive/MyDrive/Trabajo_Cualitativo/Data/02-15-2018.csv
Resultados: /content/drive/MyDrive/Trabajo_Cualitativo/results


2026/07/24 06:43:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/24 06:43:52 INFO mlflow.store.db.utils: Updating database tables
2026/07/24 06:43:56 INFO mlflow.tracking.fluent: Experiment with name 'SOC_Autonomous_Threat_Hunting_DL' does not exist. Creating a new experiment.


N features: 66 | Train: (624328, 66) | Val: (208110, 66) | Test: (208110, 66)
Mapeo del target: {0: 'Benign', 1: 'Attack / Non-Benign'}
Distribucion original de labels:
  Benign: 996077
  DoS attacks-GoldenEye: 41508
  DoS attacks-Slowloris: 10990
Labels considerados como ataque:
  - DoS attacks-GoldenEye
  - DoS attacks-Slowloris
Distribucion train codificada (0=Benign, 1=Ataque): {0: 592829, 1: 31499}
[MLP | seed=42] F1=0.9973 PR-AUC=0.9999 ROC-AUC=1.0000 Recall=0.9952 (474.0s)
[MLP | seed=43] F1=0.9975 PR-AUC=0.9998 ROC-AUC=1.0000 Recall=0.9956 (474.1s)
[MLP | seed=44] F1=0.9972 PR-AUC=0.9998 ROC-AUC=1.0000 Recall=0.9950 (484.8s)
[MLP | seed=45] F1=0.9974 PR-AUC=0.9998 ROC-AUC=1.0000 Recall=0.9955 (472.8s)
[LSTM | seed=42] F1=0.9970 PR-AUC=0.9996 ROC-AUC=1.0000 Recall=0.9953 (1109.4s)
[LSTM | seed=43] F1=0.9828 PR-AUC=0.9945 ROC-AUC=0.9998 Recall=0.9824 (1113.3s)
[LSTM | seed=44] F1=0.9965 PR-AUC=0.9997 ROC-AUC=1.0000 Recall=0.9938 (1111.5s)
[LSTM | seed=45] F1=0.9956 PR-AUC=0.9992 